In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Dynamic path resolution for Kaggle vs Local Repo
if os.path.exists("/kaggle/input/competitions/agriculture-climate-slm-challenge/"):
    base = "/kaggle/input/competitions/agriculture-climate-slm-challenge/"
    output_dir = "/kaggle/working/"
elif os.path.exists("/kaggle/input/"):
    base = "/kaggle/input/"
    output_dir = "/kaggle/working/"
else:
    base = "../data/"
    output_dir = "../"

print(f"Using data directory: {base}")

In [ ]:
train = pd.read_csv(os.path.join(base, "train_qa.csv"))
test = pd.read_csv(os.path.join(base, "test_questions.csv"))
train.head(), test.head()

In [ ]:
# Fit vectorizer ONCE on full question corpus
vec = TfidfVectorizer(ngram_range=(1, 2))
train_matrix = vec.fit_transform(train["question"])

rows = []
for _, t in test.iterrows():
    if "topic" in test.columns and "topic" in train.columns:
        sub_mask = (train["topic"] == t["topic"])
        if sub_mask.any():
            sub = train[sub_mask]
            sub_matrix = train_matrix[sub_mask]
        else:
            sub = train
            sub_matrix = train_matrix
    else:
        sub = train
        sub_matrix = train_matrix

    test_vec = vec.transform([t["question"]])
    sims = cosine_similarity(test_vec, sub_matrix)
    best_idx = sims.argmax()
    ans = sub.iloc[best_idx]["reference_answer"]
    rows.append({"QuestionId": t["QuestionId"], "Answer": ans})

submission = pd.DataFrame(rows)
submission.to_csv(os.path.join(output_dir, "submission.csv"), index=False)
submission.head()